# El Niño 2026 — Notebook 07: Independent satellite evidence and publication figures
**le-gis.com climate project · supports the International Journal of Climatology manuscript**

The first six notebooks lean on reanalysis (ERA5, GLORYS12). This notebook checks the key links of the story with **satellite observations only**, adds the **fire impact** in Indonesia, and draws **print-quality figures** for the paper.

| Part | Link in the story | Satellite data | Output |
|---|---|---|---|
| 1 | Wind bursts launched Kelvin waves that reached South America | Sea-level anomaly from altimetry (Copernicus DUACS L4) | Hovmöller, measured wave speed, arrival dates vs prediction, agreement with GLORYS thermocline |
| 2 | Two waves of westerly wind bursts were the trigger | Scatterometer-based L4 winds (Copernicus, ASCAT) | Burst strength from satellite vs ERA5 |
| 3 | Indonesia's dry season lost about half its rain | GPM IMERG V07 + CHIRPS, same 2001–2020 base | National and provincial deficits, product agreement |
| 4 | Drought → fire | MODIS active fires (FIRMS) | Jun–Oct fire counts 2001–2026, by island and province; link to rain deficit |
| 5 | — | OISST; Notebook 03 indices | Fig. 1 (map) and Fig. 4 (event comparison) at 300 dpi |

**Runtime, first run:** about 45–90 min (altimetry climatology and scatterometer winds are the largest downloads). Everything is cached to Drive.
**Accounts:** the same Copernicus Marine and Earth Engine logins as before.

In [ ]:
!pip -q install copernicusmarine netCDF4 earthengine-api adjustText
import os, time, json, getpass, logging, warnings
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, matplotlib.colors as mcolors
from concurrent.futures import ThreadPoolExecutor
from scipy import stats
warnings.filterwarnings("ignore")
plt.rcParams.update({"axes.grid": True, "grid.alpha": .25, "font.size": 9, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "savefig.dpi": 300, "savefig.bbox": "tight", "font.family": "DejaVu Sans"})

from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/le-gis_enso"
CACHE, WEB = f"{ROOT}/cache", f"{ROOT}/web"
C7, OUT = f"{CACHE}/nb07", f"{WEB}/nb07"; FIG = f"{OUT}/figures"
[os.makedirs(d, exist_ok=True) for d in (CACHE, WEB, C7, OUT, FIG)]
TODAY = pd.Timestamp.today().normalize(); this_year = TODAY.year

# ---- knobs ----
T0 = pd.Timestamp("2025-07-01")               # start of the 2025-26 Hovmöller window
CLIM = (1993, 2020)                           # altimetry climatology (DUACS starts 1993)
RAIN_BASE, RAIN_MONTHS = (2001, 2020), [6, 7, 8, 9]   # IMERG starts mid-2000, so the common rainfall base is 2001-2020
FIRE_CONF = 50                                # MODIS fire confidence threshold (%)
KELVIN = 2.5                                  # first-mode Kelvin wave speed used for predicted paths (m/s)
EVENTS = [1997, 2015, 2023, 2026]
MAP_MONTH = "2026-09"                         # Fig. 1: last month of the study period (onset study, Jul 2025 - Sep 2026)
COLORS = {1997: "#1f77b4", 2015: "#2ca02c", 2023: "#9467bd", 2026: "#dc2626"}
IDN = dict(west=94.5, east=141.5, south=-11.5, north=6.5)

def save(fig, name):
    for ext in ("png", "pdf"): fig.savefig(f"{FIG}/{name}.{ext}")
    print("saved", f"{FIG}/{name}.png")

def deg_axes(ax):
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}°E"))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: "0°" if v == 0 else f"{abs(v):.0f}°{'N' if v > 0 else 'S'}"))

DIRS = {"Utara": "North", "Selatan": "South", "Tengah": "Central", "Barat": "West", "Timur": "East", "Tenggara": "Southeast"}
def short(p):   # GAUL province name -> English short name, e.g. "Kalimantan Tengah" -> "Central Kalimantan"
    w = p.replace("Sumatera", "Sumatra").replace("Jawa", "Java").replace("Kepulauan ", "").split()
    return f"{DIRS[w[-1]]} {' '.join(w[:-1])}" if len(w) > 1 and w[-1] in DIRS else " ".join(w)

In [ ]:
import copernicusmarine as cm, ee
CM_USER = input("Copernicus Marine username: ").strip()
CM_PASS = getpass.getpass("Copernicus Marine password: ")
assert cm.login(username=CM_USER, password=CM_PASS, force_overwrite=True), "Copernicus Marine login failed: check username/password, then re-run this cell"
CMK = dict(username=CM_USER, password=CM_PASS)   # passed to every download, so the toolbox never asks again
print("Copernicus Marine: logged in as", CM_USER)
EE_PROJECT = "YOUR-EE-PROJECT"   # <-- same Earth Engine project as before
ee.Authenticate(); ee.Initialize(project=EE_PROJECT)
for n in ("copernicusmarine", "urllib3.connectionpool"): logging.getLogger(n).setLevel(logging.ERROR)

## Shared tools

In [ ]:
def with_retry(f, tries=4, wait=15):
    for k in range(tries):
        try: return f()
        except Exception as e:
            if k == tries - 1 or "exceed" in str(e) or "not available" in str(e).lower(): raise   # wrong date range: retrying will not help
            print("   retry:", str(e)[:90]); time.sleep(wait * (k + 1))
def clean_time(x):
    x = x.sortby("time"); _, k = np.unique(x.time.values, return_index=True); return x.isel(time=np.sort(k))

def find_ids(product_id, must):
    try:
        cat = cm.describe(product_id=product_id)
        prods = cat.products if hasattr(cat, "products") else cat["products"]
        ids = [(d.dataset_id if hasattr(d, "dataset_id") else d["dataset_id"]) for p in prods for d in (p.datasets if hasattr(p, "datasets") else p["datasets"])]
        return [i for i in ids if all(m in i for m in must)]
    except Exception as e:
        print("   describe failed:", str(e)[:70]); return []

def cm_strip(ds_id, var, t0, t1, lat, pieces, step=1):
    # mean over latitude, longitude 0-360, ~1 degree; pieces = [(west, east), ...] in -180..180
    out = []
    for a, b in pieces:
        kw = dict(dataset_id=ds_id, variables=[var], **CMK, minimum_longitude=a, maximum_longitude=b, minimum_latitude=lat[0], maximum_latitude=lat[1],
                  start_datetime=str(pd.Timestamp(t0).date()), end_datetime=str(pd.Timestamp(t1).date()))
        try: ds = cm.open_dataset(**kw, coordinates_selection_method="inside")    # take whatever part of the dates the dataset covers
        except TypeError: ds = cm.open_dataset(**kw)                               # older toolbox without that option
        da = ds[var]
        if "depth" in da.dims: da = da.isel(depth=0, drop=True)
        if step > 1: da = da.isel(time=slice(None, None, step))
        da = da.rename({k: v for k, v in {"latitude": "lat", "longitude": "lon"}.items() if k in da.dims}).mean("lat").load()
        out.append(da)
    da = xr.concat(out, "lon"); da = da.assign_coords(lon=da.lon % 360).sortby("lon")
    da = da.groupby(np.floor(da.lon) + 0.5).mean().rename({"floor": "lon"}) if "floor" in da.dims else da.groupby_bins("lon", np.arange(np.floor(float(da.lon.min())), np.ceil(float(da.lon.max())) + 1, 1.0)).mean()
    if "lon_bins" in da.dims: da = da.assign_coords(lon_bins=[iv.mid for iv in da.lon_bins.values]).rename({"lon_bins": "lon"})
    return da

def load_web_json(name):
    # earlier exports: look anywhere under the project folder, otherwise ask for the file (it is also in the website repo)
    import glob
    hits = sorted(glob.glob(f"{ROOT}/**/{name}", recursive=True), key=os.path.getmtime, reverse=True)
    if not hits:
        from google.colab import files
        print(f"{name} not found on Drive. Upload it (website repo: apps/web/public/project/elnino/{name})")
        up = files.upload(); assert name in up, f"please upload {name}"
        open(f"{WEB}/{name}", "wb").write(up[name]); hits = [f"{WEB}/{name}"]
    print("using", hits[0]); return json.load(open(hits[0]))

def ee_region(img, bands, west, east, south, north, res):
    # Earth Engine image -> numpy arrays on a regular grid (pixel centres); west/east in 0-360, split at the dateline
    pieces = [(west, east)] if east <= 180 else ([(west - 360, east - 360)] if west >= 180 else [(west, 180), (-180, east - 360)])
    h = int(round((north - south) / res)); parts = []
    for a, b in pieces:
        w = int(round((b - a) / res))
        arr = with_retry(lambda: ee.data.computePixels({"expression": img, "fileFormat": "NUMPY_NDARRAY",
            "grid": {"dimensions": {"width": w, "height": h},
                     "affineTransform": {"scaleX": res, "shearX": 0, "translateX": a, "shearY": 0, "scaleY": -res, "translateY": north}, "crsCode": "EPSG:4326"}}))
        parts.append(arr)
    out = {}
    for bnd in bands:
        x = np.concatenate([np.asarray(p[bnd], dtype=float) for p in parts], axis=1)[::-1]; x[x <= -9998] = np.nan; out[bnd] = x
    lats = south + res * (np.arange(h) + .5); lons = (west + res * (np.arange(sum(int(round((b - a) / res)) for a, b in pieces)) + .5)) % 360
    return lats, lons, out

## Part 1 — Kelvin waves seen by satellite altimetry
Sea level along the equator rises over a deeper, warmer thermocline, so altimetry sees Kelvin waves directly, without an ocean model. We measure their speed and compare their arrival off South America with the arrival predicted from each wind burst (Notebook 02).

In [ ]:
ALT_MY, ALT_NRT = "SEALEVEL_GLO_PHY_L4_MY_008_047", "SEALEVEL_GLO_PHY_L4_NRT_008_046"
ALT = {"my_d": find_ids(ALT_MY, ("P1D",)) + ["cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D", "cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.25deg_P1D"],
       "my_m": find_ids(ALT_MY, ("P1M",)) + ["cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1M-m", "cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.25deg_P1M-m"],
       "nrt_d": find_ids(ALT_NRT, ("P1D",)) + ["cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.125deg_P1D", "cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.25deg_P1D"]}
EQ, PIECES = (-2, 2), [(120, 180), (-180, -79)]

def first_ok(ids, f):
    err = None
    for i in dict.fromkeys(ids):
        try: return i, f(i)
        except Exception as e: err = e
    raise RuntimeError(f"no dataset worked: {str(err)[:120]}")

f_cl = f"{C7}/sla_clim_{CLIM[0]}_{CLIM[1]}.nc"
if os.path.exists(f_cl): SLA_CLIM = xr.open_dataarray(f_cl).load()
else:
    parts = []
    for y0 in range(CLIM[0], CLIM[1] + 1, 7):                         # 7-year chunks
        y1 = min(y0 + 6, CLIM[1]); print(f"  altimetry monthly {y0}-{y1}")
        _, d = first_ok(ALT["my_m"], lambda i: with_retry(lambda: cm_strip(i, "sla", f"{y0}-01-01", f"{y1}-12-31", EQ, PIECES)))
        parts.append(d)
    m = clean_time(xr.concat(parts, "time")); SLA_CLIM = m.groupby("time.month").mean(); SLA_CLIM.to_netcdf(f_cl)

f_d = f"{C7}/sla_daily_recent.nc"
if os.path.exists(f_d) and time.time() - os.path.getmtime(f_d) < 3 * 86400: SLA = xr.open_dataarray(f_d).load()
else:
    parts = []
    for key in ("my_d", "nrt_d"):
        t0 = T0 if not parts else pd.Timestamp(parts[-1].time.values[-1]) + pd.Timedelta(days=1)
        try:
            i, d = first_ok(ALT[key], lambda i: with_retry(lambda: cm_strip(i, "sla", t0, TODAY, EQ, PIECES)))
            print(f"  {key}: {i} {str(d.time.values[0])[:10]} -> {str(d.time.values[-1])[:10]}"); parts.append(d)
        except Exception as e: print(f"  {key}: unavailable ({str(e)[:80]})")
    SLA = clean_time(xr.concat(parts, "time")); SLA.to_netcdf(f_d)
SLA_A = (SLA.groupby("time.month") - SLA_CLIM.interp(lon=SLA.lon)).drop_vars("month") * 100          # cm, vs 1993-2020 seasonal cycle
print("altimetry anomaly:", str(SLA_A.time.values[0])[:10], "->", str(SLA_A.time.values[-1])[:10], "| lon", float(SLA_A.lon.min()), "-", float(SLA_A.lon.max()))

In [ ]:
HOV = load_web_json("hov_web.json")                                     # Notebook 03 export: wind bursts + GLORYS thermocline
WWB = pd.DataFrame(HOV["wwb"]); WWB["start"] = pd.to_datetime(WWB.start)
trig = WWB[WWB.start < "2026-05-01"].copy()                                     # trigger bursts (Notebook 02)
trig["pred_arrival"] = trig.start + pd.to_timedelta((270 - trig.lon) * 111.32e3 / KELVIN / 86400, unit="D")

hp = SLA_A - SLA_A.rolling(time=61, center=True, min_periods=31).mean()          # intraseasonal part: isolates the waves
west, east = hp.sel(lon=slice(160, 170)).mean("lon"), hp.sel(lon=slice(260, 270)).mean("lon")
win = slice("2025-12-01", "2026-09-30"); w_, e_ = west.sel(time=win).to_series(), east.sel(time=win).to_series()
# Wave speed by ridge tracking: follow each wave crest eastward without assuming its speed.
# Start near the burst, then at every 5° step look for the crest 1-7 days after the previous one (allowed speed range ~0.9-6 m/s).
HP3 = hp.rolling(time=5, center=True, min_periods=3).mean()
def track(b, step=5):
    lo0 = 5 * np.ceil((b.lon + 5) / 5); s = HP3.sel(lon=lo0, method="nearest").sel(time=slice(b.start, b.start + pd.Timedelta(days=25))).to_series().dropna()
    if not len(s): return None
    pts = [(lo0, s.idxmax(), s.max())]
    for lo in np.arange(lo0 + step, 266, step):
        t_prev = pts[-1][1]; s = HP3.sel(lon=lo, method="nearest").sel(time=slice(t_prev + pd.Timedelta(days=1), t_prev + pd.Timedelta(days=7))).to_series().dropna()
        if not len(s) or s.max() <= 0: break
        pts.append((lo, s.idxmax(), s.max()))
    return pd.DataFrame(pts, columns=["lon", "time", "cm"])
trk, sp = {}, []
for _, b in trig.iterrows():
    p = track(b)
    if p is None or len(p) < 8: continue
    x = (p.lon.values - p.lon.values[0]) * 111.32e3; tt = (p.time - p.time.iloc[0]).dt.total_seconds().values
    f = stats.linregress(tt, x); trk[b.start] = p
    sp.append({"burst start": b.start.date(), "tracked to (°E)": float(p.lon.iloc[-1]), "speed (m/s)": round(f.slope, 2), "fit r": round(f.rvalue, 2), "mean crest (cm)": round(float(p.cm.mean()), 1)})
SP = pd.DataFrame(sp); SP.to_csv(f"{OUT}/P1_kelvin_speeds.csv", index=False); display(SP)
SPEED = float(SP["speed (m/s)"].median()) if len(SP) else np.nan
SPEED_RANGE = (float(SP["speed (m/s)"].min()), float(SP["speed (m/s)"].max())) if len(SP) else (np.nan, np.nan)
print(f"Kelvin wave speed (ridge tracking, median of {len(SP)} waves): {SPEED:.2f} m/s, range {SPEED_RANGE[0]:.2f}-{SPEED_RANGE[1]:.2f} (theory, first baroclinic mode: 2.4-2.8 m/s)")

rows = []
for _, b in trig.iterrows():
    s = e_[(e_.index >= b.pred_arrival - pd.Timedelta(days=25)) & (e_.index <= b.pred_arrival + pd.Timedelta(days=25))]
    if len(s): rows.append({"burst start": b.start.date(), "burst lon (°E)": b.lon, "peak wind (m/s)": b.peak, "predicted arrival 95°W": b.pred_arrival.date(),
                            "observed sea-level peak": s.idxmax().date(), "difference (days)": (s.idxmax() - b.pred_arrival).days, "peak anomaly (cm)": round(float(s.max()), 1)})
ARR = pd.DataFrame(rows); ARR.to_csv(f"{OUT}/P1_kelvin_arrivals.csv", index=False); display(ARR)

# agreement with the GLORYS thermocline (Notebook 03 weekly Hovmöller)
d20 = pd.DataFrame(HOV["vars"]["d20"], index=pd.to_datetime(HOV["time"]), columns=HOV["lon"]).astype(float)
wk = SLA_A.resample(time="7D").mean().interp(lon=d20.columns.values).to_pandas()
wk = wk.reindex(d20.index, method="nearest", tolerance=pd.Timedelta(days=4))
j = pd.DataFrame({"sla": wk.values.ravel(), "d20": d20.values.ravel()}).dropna()
R_D20 = j.sla.corr(j.d20); print(f"Pattern correlation, altimetry vs GLORYS thermocline depth (weekly, 2°S-2°N): r = {R_D20:.2f}, n = {len(j)}")

In [ ]:
fig, (ax, axr) = plt.subplots(1, 2, figsize=(10, 6.5), gridspec_kw={"width_ratios": [3, 1]}, sharey=True)
A = SLA_A.sel(time=slice(T0, None)).resample(time="3D").mean()
t = pd.to_datetime(A.time.values)
pc = ax.pcolormesh(A.lon, t, A.values, cmap="RdBu_r", norm=mcolors.TwoSlopeNorm(0, -25, 25), shading="auto")
for p in trk.values(): ax.plot(p.lon, p.time, c="#facc15", lw=1.6)                  # tracked crests
for _, b in trig.iterrows():
    ax.plot([b.lon, 270], [b.start, b.pred_arrival], "k--", lw=1)
    ax.plot(b.lon, b.start, marker=">", c="k", ms=6)
ax.set_xlim(120, 280); ax.set_xticks([150, 180, 210, 240, 270]); ax.set_xticklabels(["150°E", "180°", "150°W", "120°W", "90°W"])
ax.invert_yaxis(); ax.set_title(f"(a) Sea-level anomaly, 2°S–2°N (satellite altimetry)\nyellow: tracked crests, median {SPEED:.1f} m/s; dashed: {KELVIN} m/s prediction")
plt.colorbar(pc, ax=ax, orientation="horizontal", pad=.06, fraction=.04, label="cm vs 1993–2020")
axr.plot(east.sel(time=t, method="nearest").values, t, c="#dc2626", lw=1.2); axr.axvline(0, c="k", lw=.7)
for _, b in trig.iterrows(): axr.axhline(b.pred_arrival, c="k", ls=":", lw=.8)
axr.set_title("(b) 90–100°W, intraseasonal"); axr.set_xlabel("cm")
plt.tight_layout(); save(fig, "F_altimetry_kelvin"); plt.show()

json.dump({"lon": [float(x) for x in SLA_A.lon.values], "time": [str(x)[:10] for x in A.time.values],
           "sla": [[None if not np.isfinite(v) else round(float(v), 1) for v in row] for row in A.interp(lon=SLA_A.lon).values],
           "speed": round(float(SPEED), 2), "r_d20": round(float(R_D20), 2)}, open(f"{WEB}/sla_hov.json", "w"), separators=(",", ":"))

## Part 2 — Westerly wind bursts seen by satellite scatterometers
Copernicus L4 winds are built from ASCAT scatterometer observations (ERA5 only fills gaps), so they are an independent check of the burst strength used in Notebook 02.

In [ ]:
W_MY, W_NRT = "WIND_GLO_PHY_L4_MY_012_006", "WIND_GLO_PHY_L4_NRT_012_004"
def wind_ids(prod, tag):
    d = find_ids(prod, ("P1D",)); h = find_ids(prod, ("PT1H",))
    return [(i, 1) for i in d] + [(i, 6) for i in h] + [(f"cmems_obs-wind_glo_phy_{tag}_l4_0.125deg_PT1H", 6), (f"cmems_obs-wind_glo_phy_{tag}_l4_0.25deg_PT1H", 6)]
WIDS = {"my": wind_ids(W_MY, "my"), "nrt": wind_ids(W_NRT, "nrt")}
WSTRIP, WPIECES = (-5, 5), [(130, 180), (-180, -150)]
months = pd.date_range("2025-11-01", TODAY, freq="MS")

def wind_month(m):
    f = f"{C7}/scat_u_{m:%Y_%m}.nc"
    if os.path.exists(f) and m + pd.offsets.MonthEnd(0) < TODAY - pd.Timedelta(days=7): return xr.open_dataarray(f).load()
    t1 = min(m + pd.offsets.MonthEnd(0), TODAY)
    for key in ("my", "nrt"):
        for i, step in dict.fromkeys(WIDS[key]):
            try:
                d = cm_strip(i, "eastward_wind", m, t1, WSTRIP, WPIECES, step=step)
                d = d.resample(time="1D").mean(); d.to_netcdf(f); return d
            except Exception: continue
    return None
SCAT = [d for d in (wind_month(m) for m in months) if d is not None]
SCAT = clean_time(xr.concat(SCAT, "time")) if SCAT else None
print("scatterometer winds:", None if SCAT is None else (str(SCAT.time.values[0])[:10], str(SCAT.time.values[-1])[:10]))

ERA5H = ee.ImageCollection("ECMWF/ERA5/HOURLY").select("u_component_of_wind_10m")
def era_day(d):
    img = ERA5H.filterDate(str(d.date()), str((d + pd.Timedelta(days=1)).date())).mean().unmask(-9999, False).toFloat()
    _, lons, a = ee_region(img, ["u_component_of_wind_10m"], 130, 210, -5, 5, 1.0); return lons, np.nanmean(a["u_component_of_wind_10m"], axis=0)
f_e = f"{C7}/era5_u_strip.nc"
days = pd.date_range("2025-11-01", min(TODAY - pd.Timedelta(days=7), pd.Timestamp(SCAT.time.values[-1]) if SCAT is not None else TODAY), freq="D")
ERA = xr.open_dataarray(f_e).load() if os.path.exists(f_e) else None
need = days if ERA is None else days[days > pd.Timestamp(ERA.time.values[-1])]
if len(need):
    with ThreadPoolExecutor(max_workers=10) as ex: res = list(ex.map(era_day, need))
    new = xr.DataArray(np.stack([r[1] for r in res]), dims=("time", "lon"), coords={"time": need, "lon": res[0][0]})
    ERA = new if ERA is None else clean_time(xr.concat([ERA, new], "time")); ERA.to_netcdf(f_e)

S = SCAT.interp(lon=ERA.lon).sel(time=ERA.time, method="nearest"); S["time"] = ERA.time
j = pd.DataFrame({"scat": S.values.ravel(), "era5": ERA.values.ravel()}).dropna()
R_W, BIAS_W = j.scat.corr(j.era5), float((j.scat - j.era5).mean())
rows = []
for _, b in WWB.iterrows():
    sl = dict(time=slice(b.start, pd.Timestamp(b.end) if "end" in b else b.start), lon=slice(b.lon - 5, b.lon + 5))
    rows.append({"burst start": b.start.date(), "lon (°E)": b.lon, "ERA5 mean u (m/s)": float(ERA.sel(**sl).mean()), "scatterometer mean u (m/s)": float(S.sel(**sl).mean())})
BUR = pd.DataFrame(rows).round(2); BUR.to_csv(f"{OUT}/P2_bursts_satellite_vs_era5.csv", index=False)
print(f"Daily 10 m zonal wind, 5°S-5°N, 130°E-150°W: r = {R_W:.2f}, mean difference scatterometer - ERA5 = {BIAS_W:+.2f} m/s, n = {len(j)}"); display(BUR)

# The L4 product is ERA5 corrected with scatterometer data, so its r with ERA5 is high by construction.
# The truly independent check is the L3 product: ASCAT swath winds only (MetOp-B/C, gaps between passes).
L3IDS = find_ids("WIND_GLO_PHY_L3_MY_012_005", ("ascat",)) + find_ids("WIND_GLO_PHY_L3_NRT_012_002", ("ascat",))
L3IDS = [i for i in L3IDS if "0.125deg" in i] or L3IDS
print("ASCAT L3 datasets:", L3IDS)
f3 = f"{C7}/ascat_l3_u.nc"
if os.path.exists(f3) and time.time() - os.path.getmtime(f3) < 3 * 86400: S3 = xr.open_dataarray(f3).load()
else:
    parts = []
    for i in L3IDS:
        try: d = with_retry(lambda: cm_strip(i, "eastward_wind", "2025-11-01", TODAY, WSTRIP, WPIECES), tries=2); parts.append(d.resample(time="1D").mean().interp(lon=ERA.lon)); print("   ok", i)
        except Exception as e: print("   skip", i, str(e)[:70])
    S3 = xr.concat(parts, "src", join="outer").mean("src") if parts else None
    if S3 is not None: S3.to_netcdf(f3)
if S3 is not None:
    S3 = S3.reindex(time=ERA.time, method="nearest", tolerance=np.timedelta64(12, "h"))
    j3 = pd.DataFrame({"a": S3.values.ravel(), "e": ERA.values.ravel()}).dropna()
    R_W3, BIAS_W3, COV3 = j3.a.corr(j3.e), float((j3.a - j3.e).mean()), len(j3) / ERA.size
    BUR["ASCAT L3 mean u (m/s)"] = [round(float(S3.sel(time=slice(b.start, pd.Timestamp(b.end)), lon=slice(b.lon - 5, b.lon + 5)).mean()), 2) for _, b in WWB.iterrows()]
    BUR.to_csv(f"{OUT}/P2_bursts_satellite_vs_era5.csv", index=False); display(BUR)
    print(f"ASCAT L3 swaths vs ERA5 (independent): r = {R_W3:.2f}, mean difference {BIAS_W3:+.2f} m/s, coverage {100 * COV3:.0f} % of day-cells")
else: R_W3 = BIAS_W3 = np.nan

panels = ([(S3, "(a) ASCAT swaths (L3)")] if S3 is not None else []) + [(S, f"({'b' if S3 is not None else 'a'}) Copernicus L4"), (ERA, f"({'c' if S3 is not None else 'b'}) ERA5")]
fig, axs = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 5.8), sharey=True)
for ax, (da, lab) in zip(axs, panels):
    pc = ax.pcolormesh(da.lon, pd.to_datetime(da.time.values), da.values, cmap="RdBu_r", norm=mcolors.TwoSlopeNorm(0, -12, 12), shading="auto")
    ax.scatter(WWB.lon, WWB.start, marker=">", c="k", s=18, zorder=5)
    ax.set_title(lab); ax.set_xticks([140, 160, 180, 200]); ax.set_xticklabels(["140°E", "160°E", "180°", "160°W"])
axs[0].invert_yaxis(); plt.colorbar(pc, ax=axs, orientation="horizontal", fraction=.04, pad=.08, label="10 m zonal wind, 5°S–5°N (m/s; positive = westerly)")
fig.suptitle(f"Westerly wind bursts in satellite winds and ERA5 (▶ bursts from Notebook 02)", fontweight="bold", y=1.0)
fig.text(.5, .955, f"vs ERA5, daily: ASCAT L3 r = {R_W3:.2f} ({BIAS_W3:+.2f} m/s), L4 r = {R_W:.2f} ({BIAS_W:+.2f} m/s)", ha="center", fontsize=9, color="#475569")
save(fig, "F_winds_scatterometer"); plt.show()

## Part 3 — Indonesian rainfall: two independent products
GPM IMERG (satellite microwave + infrared, gauge-calibrated) and CHIRPS (infrared + stations) on the same 2001–2020 base and the same June–September window.

In [ ]:
GAUL = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq("ADM0_NAME", "Indonesia"))
PROV = GAUL.aggregate_array("ADM1_NAME").getInfo(); NP = len(PROV)
gl = GAUL.toList(NP)
PID_IMG = ee.FeatureCollection(ee.List.sequence(0, NP - 1).map(lambda i: ee.Feature(gl.get(i)).set("pid", i))).reduceToImage(["pid"], ee.Reducer.first()).rename("pid").unmask(-9999).toFloat()
def pid_grid(res):
    lats, lons, a = ee_region(PID_IMG, ["pid"], IDN["west"], IDN["east"], IDN["south"], IDN["north"], res); return lats, lons, a["pid"]
LAT1, LON1, PID1 = pid_grid(0.1)
def island(name):
    n = name.lower()
    for key, isl in [("kalimantan", "Kalimantan"), ("sulawesi", "Sulawesi"), ("gorontalo", "Sulawesi"), ("papua", "Papua"), ("irian", "Papua"),
                     ("maluku", "Maluku"), ("nusa", "Nusa Tenggara & Bali"), ("bali", "Nusa Tenggara & Bali"), ("jawa", "Java"), ("jakarta", "Java"),
                     ("banten", "Java"), ("yogya", "Java")]:
        if key in n: return isl
    return "Sumatra"
ISL = {i: island(p) for i, p in enumerate(PROV)}

IM_M = ee.ImageCollection("NASA/GPM_L3/IMERG_MONTHLY_V07"); IM_H = ee.ImageCollection("NASA/GPM_L3/IMERG_V07"); CH = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD")
def month_imerg(y, m, use_monthly=True):
    t0 = ee.Date.fromYMD(y, m, 1); t1 = t0.advance(1, "month"); hours = t1.difference(t0, "hour")
    src = IM_M if use_monthly else IM_H
    return src.filterDate(t0, t1).select("precipitation").mean().multiply(hours)                 # mm/hr x hours = mm
def month_chirps(y, m):
    t0 = ee.Date.fromYMD(y, m, 1); return CH.filterDate(t0, t0.advance(1, "month")).select("precipitation").sum()
def season(fn, y, **kw): return ee.ImageCollection([fn(y, m, **kw) for m in RAIN_MONTHS]).sum()
MONTHLY_OK = IM_M.filterDate(f"{RAIN_BASE[0]}-01-01", f"{RAIN_BASE[1]}-12-31").size().getInfo() > 200
last_chirps = pd.Timestamp(CH.sort("system:time_start", False).first().date().format("YYYY-MM-dd").getInfo())
print("IMERG monthly collection usable for the base:", MONTHLY_OK, "| CHIRPS latest pentad:", last_chirps.date())

def rain_grids():
    f = f"{C7}/rain_{this_year}.npz"
    if os.path.exists(f) and time.time() - os.path.getmtime(f) < 7 * 86400: return dict(np.load(f))
    base_i = ee.ImageCollection([season(month_imerg, y, use_monthly=MONTHLY_OK) for y in range(RAIN_BASE[0], RAIN_BASE[1] + 1)]).mean()
    now_i = season(month_imerg, this_year, use_monthly=False)                                      # 2026: half-hourly (provisional) run
    base_c = ee.ImageCollection([season(month_chirps, y) for y in range(RAIN_BASE[0], RAIN_BASE[1] + 1)]).mean()
    now_c = season(month_chirps, this_year)
    img = ee.Image.cat([base_i.rename("bi"), now_i.rename("ni"), base_c.rename("bc"), now_c.rename("nc")]).unmask(-9999).toFloat()
    _, _, a = ee_region(img, ["bi", "ni", "bc", "nc"], IDN["west"], IDN["east"], IDN["south"], IDN["north"], 0.1)
    np.savez(f, **a); return a
RG = rain_grids()
land = (PID1 >= 0) & np.isfinite(PID1)
def pct(now, base):
    with np.errstate(invalid="ignore", divide="ignore"): p = 100 * (now / base - 1)
    p[(base < 50) | ~land] = np.nan; return p
P_IM, P_CH = pct(RG["ni"], RG["bi"]), pct(RG["nc"], RG["bc"])
w = np.cos(np.deg2rad(LAT1))[:, None] * np.ones_like(PID1)
def agg(now, base, mask):
    m = mask & land & (base >= 50) & np.isfinite(now) & np.isfinite(base)
    return 100 * ((now * w)[m].sum() / (base * w)[m].sum() - 1) if m.any() else np.nan
NAT = {"IMERG": agg(RG["ni"], RG["bi"], land), "CHIRPS": agg(RG["nc"], RG["bc"], land)}
rows = []
for i, p in enumerate(PROV):
    mk = PID1 == i
    rows.append({"province": p, "island": ISL[i], "IMERG %": agg(RG["ni"], RG["bi"], mk), "CHIRPS %": agg(RG["nc"], RG["bc"], mk)})
RP = pd.DataFrame(rows).dropna(); RP.round(1).to_csv(f"{OUT}/P3_rain_provinces.csv", index=False)
R_RAIN = RP["IMERG %"].corr(RP["CHIRPS %"])
ISLR = pd.DataFrame({isl: {"IMERG %": agg(RG["ni"], RG["bi"], np.isin(PID1, [i for i in ISL if ISL[i] == isl])),
                           "CHIRPS %": agg(RG["nc"], RG["bc"], np.isin(PID1, [i for i in ISL if ISL[i] == isl]))} for isl in sorted(set(ISL.values()))}).T
print(f"Indonesia, Jun-Sep {this_year} vs {RAIN_BASE[0]}-{RAIN_BASE[1]}: IMERG {NAT['IMERG']:+.0f} %, CHIRPS {NAT['CHIRPS']:+.0f} %; provinces r = {R_RAIN:.2f} (n = {len(RP)})")
display(ISLR.round(0))

fig = plt.figure(figsize=(11, 7), layout="constrained"); gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.15])
maps = []
for k, (P, lab) in enumerate([(P_IM, "(a) GPM IMERG"), (P_CH, "(b) CHIRPS")]):
    ax = fig.add_subplot(gs[0, k]); pc = ax.pcolormesh(LON1, LAT1, P, cmap="BrBG", vmin=-80, vmax=80, shading="auto"); maps.append(ax)
    ax.set_facecolor("#e2e8f0"); ax.set_aspect("equal"); deg_axes(ax); ax.set_title(lab)
fig.colorbar(pc, ax=maps, location="bottom", shrink=.45, aspect=35, label=f"Jun–Sep {this_year} rainfall, % difference from {RAIN_BASE[0]}–{RAIN_BASE[1]} mean")
ax = fig.add_subplot(gs[1, 0]); ax.scatter(RP["CHIRPS %"], RP["IMERG %"], s=18, c="#2563eb"); ax.plot([-100, 60], [-100, 60], "k:", lw=.8)
ax.set_xlabel("CHIRPS rainfall difference (%)"); ax.set_ylabel("IMERG rainfall difference (%)"); ax.set_title(f"(c) Provinces, IMERG vs CHIRPS: r = {R_RAIN:.2f}")
ax.text(.03, .95, "each dot = one province\ndotted line = perfect agreement", transform=ax.transAxes, va="top", fontsize=8, color="#475569")
ax = fig.add_subplot(gs[1, 1]); x = np.arange(len(ISLR)); ax.bar(x - .2, ISLR["IMERG %"], .4, label="IMERG", color="#0ea5e9"); ax.bar(x + .2, ISLR["CHIRPS %"], .4, label="CHIRPS", color="#a16207")
ax.set_xticks(x); ax.set_xticklabels(ISLR.index, rotation=30, ha="right"); ax.axhline(0, c="k", lw=.7); ax.legend(fontsize=8); ax.set_title("(d) Island groups"); ax.set_ylabel("rainfall difference (%)")
save(fig, "F_rainfall_imerg_chirps"); plt.show()

## Part 4 — Fires: MODIS active-fire detections (FIRMS), June to mid-October, 2001–2026
Same calendar window every year (1 June to the latest available day), fire pixels with confidence ≥ 50 %, counted per province. FIRMS starts in November 2000, so 1997 cannot be included.

In [ ]:
FIRMS = ee.ImageCollection("FIRMS").select("confidence")
FIRE_END = min(pd.Timestamp(FIRMS.sort("system:time_start", False).first().date().format("YYYY-MM-dd").getInfo()), TODAY - pd.Timedelta(days=1))
MMDD = FIRE_END.strftime("%m-%d"); print("FIRMS latest day:", FIRE_END.date(), "| window: 1 June to", MMDD)
PROJ = FIRMS.first().projection()
LAT5, LON5, PID5 = pid_grid(0.05)

def fire_year(y):
    f = f"{C7}/fires_{y}_{MMDD}.npy"
    if os.path.exists(f) and y < this_year: return np.load(f)
    col = FIRMS.filterDate(f"{y}-06-01", str((pd.Timestamp(f"{y}-{MMDD}") + pd.Timedelta(days=1)).date())).map(lambda i: i.gte(FIRE_CONF).selfMask())
    img = col.count().unmask(0).setDefaultProjection(PROJ).reduceResolution(ee.Reducer.sum(), maxPixels=1024).rename("n").toFloat()
    _, _, a = ee_region(img, ["n"], IDN["west"], IDN["east"], IDN["south"], IDN["north"], 0.05)
    n = np.nan_to_num(a["n"]); np.save(f, n); return n
YEARS = list(range(2001, this_year + 1))
with ThreadPoolExecutor(max_workers=6) as ex: FG = dict(zip(YEARS, ex.map(fire_year, YEARS)))
rows = []
for y in YEARS:
    for i, p in enumerate(PROV): rows.append({"year": y, "province": p, "island": ISL[i], "fires": float(FG[y][PID5 == i].sum())})
FP = pd.DataFrame(rows); FP.to_csv(f"{OUT}/P4_fires_provinces.csv", index=False)
FY = FP.pivot_table(index="year", columns="island", values="fires", aggfunc="sum"); FY["Indonesia"] = FY.sum(axis=1)
base = FY.loc[RAIN_BASE[0]:RAIN_BASE[1]].mean(); RANK = int((FY["Indonesia"] > FY.loc[this_year, "Indonesia"]).sum() + 1)
print(f"Fires 1 Jun-{MMDD} {this_year}: {FY.loc[this_year, 'Indonesia']:.0f} (normal {base['Indonesia']:.0f}, x{FY.loc[this_year, 'Indonesia'] / base['Indonesia']:.1f}); rank {RANK} of {len(FY)} years")
display(FY.loc[[y for y in (2015, 2019, 2023, this_year) if y in FY.index]].round(0))

# rainfall deficit vs fire anomaly, by province (CHIRPS, same window)
fa = FP[FP.year == this_year].set_index("province").fires / FP[FP.year.between(*RAIN_BASE)].groupby("province").fires.mean()
J = RP.set_index("province").join(fa.rename("fire ratio")).join(FP[FP.year.between(*RAIN_BASE)].groupby("province").fires.mean().rename("normal fires"))
J = J[J["normal fires"] >= 10].dropna()
RHO, P_RHO = stats.spearmanr(J["CHIRPS %"], J["fire ratio"]) if len(J) > 5 else (np.nan, np.nan)
print(f"Provinces with >= 10 normal fires: Spearman rho (rain anomaly vs fire ratio) = {RHO:.2f} (p = {P_RHO:.3f}, n = {len(J)})")

fig = plt.figure(figsize=(11, 7.4), layout="constrained"); gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], width_ratios=[1.3, 1])
ax = fig.add_subplot(gs[0, :]); isl_order = FY.drop(columns="Indonesia").loc[RAIN_BASE[0]:].mean().sort_values(ascending=False).index
bottom = np.zeros(len(FY)); pal = plt.cm.tab10.colors
for k, isl in enumerate(isl_order):
    ax.bar(FY.index, FY[isl], bottom=bottom, color=pal[k % 10], label=isl, width=.8); bottom += FY[isl].values
for y in (2015, 2019, 2023, this_year):
    if y in FY.index: ax.annotate(str(y), (y, FY.loc[y, "Indonesia"]), ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_title(f"(a) MODIS fire detections in Indonesia, 1 June – {pd.Timestamp('2000-' + MMDD):%d %B} of each year"); ax.legend(ncol=4, fontsize=8); ax.set_ylabel("fire pixels")
ax = fig.add_subplot(gs[1, 0]); D = np.where(FG[this_year] > 0, FG[this_year], np.nan)
ax.set_facecolor("#e2e8f0"); ax.contourf(LON1, LAT1, np.where(land, 1, np.nan), levels=[0, 2], colors=["#f8fafc"])
pc = ax.pcolormesh(LON5, LAT5, D, cmap="inferno_r", norm=mcolors.LogNorm(1, max(10, np.nanmax(D))), shading="auto"); ax.set_aspect("equal"); deg_axes(ax)
fig.colorbar(pc, ax=ax, location="bottom", shrink=.7, aspect=30, label="fire pixels per 0.05° cell"); ax.set_title(f"(b) Where the fires were, 1 Jun – {pd.Timestamp('2000-' + MMDD):%d %b} {this_year}")
ax = fig.add_subplot(gs[1, 1]); ax.set_yscale("log"); ax.axhline(1, c="k", lw=.7)
for k, isl in enumerate(isl_order):                       # same island colours as panel (a)
    q = J[J.island == isl]
    if len(q): ax.scatter(q["CHIRPS %"], q["fire ratio"], s=30, color=pal[k % 10], edgecolor="white", lw=.5, zorder=3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}×"))
ax.set_xlabel(f"Jun–Sep {this_year} rainfall, % difference from normal (CHIRPS)"); ax.set_ylabel(f"fires in {this_year} ÷ {RAIN_BASE[0]}–{RAIN_BASE[1]} average\n(1× = normal season)")
ax.set_title(f"(c) Drier provinces burned more: ρ = {RHO:.2f}")
TOP = FP[FP.year == this_year].nlargest(5, "fires").province          # label only the five provinces with the most fires in 2026
texts = [ax.text(J.loc[p, "CHIRPS %"], J.loc[p, "fire ratio"], short(p), fontsize=7.5, color="#1e293b", fontweight="bold") for p in TOP if p in J.index]
try:
    from adjustText import adjust_text; adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="#94a3b8", lw=.5))
except Exception: pass
# caption: dots coloured by island as in (a); labelled = the five provinces with most fires in 2026
save(fig, "F_fires"); plt.show()
json.dump({"years": YEARS, "islands": {isl: [round(float(v)) for v in FY[isl].values] for isl in FY.columns}, "window_end": MMDD,
           "normal": round(float(base["Indonesia"])), "rank": RANK}, open(f"{WEB}/fires_years.json", "w"), separators=(",", ":"))

## Part 5 — Publication figures: Fig. 1 (map) and Fig. 4 (event comparison)

In [ ]:
OI = ee.ImageCollection("NOAA/CDR/OISST/V2_1").select("sst")
last_oi = pd.Timestamp(OI.sort("system:time_start", False).first().date().format("YYYY-MM-dd").getInfo())
MAPM = min(pd.Timestamp(MAP_MONTH + "-01"), last_oi.normalize().replace(day=1) - pd.DateOffset(months=1 if last_oi.day < 28 else 0))   # a complete month
now = OI.filterDate(str(MAPM.date()), str((MAPM + pd.offsets.MonthBegin(1)).date())).mean()
nrm = OI.filter(ee.Filter.calendarRange(1991, 2020, "year")).filter(ee.Filter.calendarRange(MAPM.month, MAPM.month, "month")).mean()
img = now.subtract(nrm).multiply(0.01).rename("a").unmask(-9999, False).toFloat()
LATM, LONM, AM = ee_region(img, ["a"], 100, 290, -30, 30, 0.25)
fig, ax = plt.subplots(figsize=(11, 4.2)); ax.set_facecolor("#cbd5e1")
pc = ax.pcolormesh(LONM, LATM, AM["a"], cmap="RdBu_r", vmin=-4, vmax=4, shading="auto")
for name, (la, lb, lo_a, lo_b), c in [("Niño-1+2", (-10, 0, 270, 280), "k"), ("Niño-3", (-5, 5, 210, 270), "k"), ("Niño-3.4", (-5, 5, 190, 240), "#334155"), ("Niño-4", (-5, 5, 160, 210), "#64748b")]:
    ax.add_patch(plt.Rectangle((lo_a, la), lo_b - lo_a, lb - la, fill=False, ec=c, lw=1.1, ls="-" if name != "Niño-3.4" else "--"))
    ax.text(lo_a + 1, lb + 0.8, name, fontsize=8, color=c)
ax.text(118, -8, "Indonesia", fontsize=9, fontweight="bold", color="#1e293b")
ax.set_xticks([120, 150, 180, 210, 240, 270]); ax.set_xticklabels(["120°E", "150°E", "180°", "150°W", "120°W", "90°W"])
ax.set_yticks([-20, 0, 20]); ax.set_yticklabels(["20°S", "0°", "20°N"]); ax.set_aspect("equal")
plt.colorbar(pc, ax=ax, fraction=.025, pad=.01, label="°C vs 1991–2020")
ax.set_title(f"Sea surface temperature anomaly, {MAPM:%B %Y} (NOAA OISST v2.1)")
save(fig, "Fig1_sst_map"); plt.show()

EA = load_web_json("events_aligned.json")
panels = [("nino12", "(a) Niño-1+2 (°C)"), ("nino34", "(b) Niño-3.4 (°C)"), ("d20_basin", "(c) Thermocline depth anomaly, equatorial basin (m)"), ("dmi", "(d) Indian Ocean Dipole (°C)")]
fig, axs = plt.subplots(2, 2, figsize=(11, 6.5), sharex=True)
for ax, (k, lab) in zip(axs.ravel(), panels):
    for y in EVENTS:
        s = pd.Series({int(str(m).rstrip("+")): v for m, v in EA[k].get(str(y), {}).items() if v is not None}).sort_index()
        if len(s): ax.plot(s.index, s.values, c=COLORS[y], lw=2.6 if y == 2026 else 1.4, label=f"{y}/{str(y + 1)[2:]}" + (" (to date)" if y == this_year else ""))
    ax.axhline(0, c="k", lw=.7); ax.axvline(0, c="#94a3b8", lw=.8, ls=":"); ax.set_title(lab)
for ax in axs[1]:
    ax.set_xticks([-12, -6, 0, 6, 12]); ax.set_xticklabels(["Jan\nyear −1", "Jul\nyear −1", "Jan\nonset year", "Jul\nonset year", "Jan\nyear +1"])
axs[0, 0].legend(fontsize=8); plt.tight_layout(); save(fig, "Fig4_event_comparison"); plt.show()

In [ ]:
lines = ["# Notebook 07 summary — satellite evidence", f"_Run {TODAY:%d %b %Y}._", "",
         f"- **Kelvin waves (altimetry):** crest-tracked speed {SPEED:.2f} m/s (median of {len(SP)} waves, range {SPEED_RANGE[0]:.2f}–{SPEED_RANGE[1]:.2f}); pattern correlation with the GLORYS thermocline r = {R_D20:.2f}."]
for _, r in ARR.iterrows():
    lines.append(f"  - burst {r['burst start']} at {r['burst lon (°E)']}°E: predicted arrival {r['predicted arrival 95°W']}, sea-level peak {r['observed sea-level peak']} ({r['difference (days)']:+d} days, {r['peak anomaly (cm)']} cm).")
lines.append(f"- **Wind bursts:** daily zonal wind vs ERA5: ASCAT L3 swaths (independent) r = {R_W3:.2f}, {BIAS_W3:+.2f} m/s; Copernicus L4 (ERA5 + scatterometer correction) r = {R_W:.2f}, {BIAS_W:+.2f} m/s.")
lines.append(f"- **Rainfall Jun–Sep {this_year} vs {RAIN_BASE[0]}–{RAIN_BASE[1]}:** IMERG {NAT['IMERG']:+.0f} %, CHIRPS {NAT['CHIRPS']:+.0f} %; provincial agreement r = {R_RAIN:.2f}.")
lines.append(f"- **Fires 1 Jun–{MMDD} {this_year}:** {FY.loc[this_year, 'Indonesia']:.0f} detections, {FY.loc[this_year, 'Indonesia'] / base['Indonesia']:.1f}× the {RAIN_BASE[0]}–{RAIN_BASE[1]} normal, rank {RANK} of {len(FY)} years; rain deficit vs fire ratio by province: Spearman ρ = {RHO:.2f} (p = {P_RHO:.3f}).")
open(f"{OUT}/summary07.md", "w").write("\n".join(lines)); print("\n".join(lines)); print("\nFigures in", FIG)

## Supplementary tables for the paper (Tables S1–S4)
Collects the CSV outputs of Notebooks 06 and 07 into one spreadsheet, `Supplementary_Tables.xlsx`, saved in the `web` folder.

In [ ]:
N6, N7 = f"{WEB}/nb06", OUT
s1 = pd.read_csv(f"{N7}/P1_kelvin_arrivals.csv").merge(pd.read_csv(f"{N7}/P1_kelvin_speeds.csv"), on="burst start")
rain = pd.read_csv(f"{N7}/P3_rain_provinces.csv")
fp = pd.read_csv(f"{N7}/P4_fires_provinces.csv").pivot_table(index="province", columns="year", values="fires")
fp["mean 2001-2020"] = fp.loc[:, 2001:2020].mean(axis=1); fp["fire ratio 2026"] = fp[this_year] / fp["mean 2001-2020"]
s2 = rain.set_index("province").join(fp, how="outer").reset_index().rename(columns={"index": "province"})
s2.columns = [str(c) for c in s2.columns]
s3 = pd.read_csv(f"{N6}/A_event_sums.csv")                                   # all layer depths
ab = f"{N6}/AB_event_budget_v2.csv" if os.path.exists(f"{N6}/AB_event_budget_v2.csv") else f"{N6}/AB_event_budget.csv"
v = pd.read_csv(ab).set_index(["box", "event"])                               # H = 50 m with the eddy term
for i, r in s3.iterrows():
    if r.H == 50 and (r.box, r.event) in v.index:
        for c in ("tendency", "residual"): s3.loc[i, c] = v.loc[(r.box, r.event), c]
        s3.loc[i, "eddy (H=50)"] = v.loc[(r.box, r.event), "eddy′ (°C)"]
        s3.loc[i, "residual after eddy (H=50)"] = v.loc[(r.box, r.event), "residual after eddy"]
s4 = pd.read_csv(f"{N7}/P2_bursts_satellite_vs_era5.csv").rename(columns={"scatterometer mean u (m/s)": "Copernicus L4 mean u (m/s)"})
xl = f"{WEB}/Supplementary_Tables.xlsx"
with pd.ExcelWriter(xl) as xw:
    for name, df, title in [("S1", s1, "Table S1. Kelvin waves from the trigger bursts"),
                            ("S2", s2, f"Table S2. Provincial Jun-Sep {this_year} rainfall anomaly (%) and MODIS fires, 1 Jun-{MMDD}, 2001-{this_year}"),
                            ("S3", s3, "Table S3. Mixed-layer heat budget, Mar-Sep, degC, by box, event and layer depth H (m)"),
                            ("S4", s4, "Table S4. Mean 10 m zonal wind over each burst, 5S-5N, m/s")]:
        pd.DataFrame([[title]]).to_excel(xw, sheet_name=name, index=False, header=False)
        df.round(3).to_excel(xw, sheet_name=name, index=False, startrow=2)
print("saved", xl)

## Reading the results
* **Altimetry:** a measured speed near 2.4–2.8 m/s and sea-level peaks within about two weeks of the predicted arrivals confirm the wind-burst → Kelvin-wave → coastal-warming chain without an ocean model.
* **Scatterometer:** the L4 product is ERA5 corrected with scatterometer data, so its agreement with ERA5 is partly by construction. The L3 ASCAT swaths are the independent test: if they show the same bursts with r above ~0.8, the burst strengths in Notebook 02 are not an ERA5 artefact.
* **Fire caveats:** MODIS Terra and Aqua orbits have drifted since about 2020 (later overpass times change what fraction of fires is seen), and some fires in southern Papua may come from large-scale land clearing as well as drought. Check the Papua jump against VIIRS (from 2012) and land-cover change before attributing it to El Niño.
* **Rainfall:** if IMERG and CHIRPS agree nationally and by province, the −46 % headline is robust to the product choice. Values here use 2001–2020 and June–September, so they differ slightly from the CHIRPS 1991–2020, 1 June–mid-October figure.
* **Fires:** the rank of 2026 among 26 years, and the province-level link between rain deficit and fires, give the impact section its second, independent indicator.

**References:** Pujol et al. 2016 *Ocean Science* (DUACS altimetry); Bentamy et al. 2021 (Copernicus L4 winds); Huffman et al. 2023 (IMERG V07 ATBD); Funk et al. 2015 *Sci. Data* (CHIRPS); Giglio et al. 2016 *Remote Sens. Environ.* (MODIS active fire); Huang et al. 2021 *J. Climate* (OISST v2.1); Field et al. 2016 *PNAS* (Indonesian fire and El Niño).